In [2]:
import sys
sys.path.append("..") # needed to import dataloader.py

import torch
import torchvision
from dataloader import MultiTaskBrainDataset
from torch.utils.data import DataLoader
from torchinfo import summary

torch.manual_seed(42) # set seed for reproducibility
device = torch.device("cpu")
device

device(type='cpu')

In [3]:
class FeedForwardWithSkip(torch.nn.Module):
    def __init__(self, input_dim, output_dim):
        super(FeedForwardWithSkip, self).__init__()
        self.linear1 = torch.nn.Linear(input_dim, output_dim)
        self.linear2 = torch.nn.Linear(output_dim, output_dim)
        self.relu = torch.nn.ReLU()
    
    def forward(self, x):
        residual = x
        x = self.linear1(x)
        x = self.relu(x)
        x = self.linear2(x)
        x = x + residual  # Skip connection
        return x


class ConvolutionWithSkipTumorClassification(torch.nn.Module):
    """
    Ultra-lightweight CNN for 4-class brain tumor classification.
    
    Optimized for CPU training without BatchNorm.
    Uses Dropout for regularization instead.
    """

    def __init__(self, num_classes=4):
        super(ConvolutionWithSkipTumorClassification, self).__init__()
        
        # Block 1: 1 -> 16 channels
        self.block1 = torch.nn.Sequential(
            torch.nn.Conv2d(1, 16, 3, stride=2, padding=1),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(2, 2),
            torch.nn.Dropout(0.25),
        )
        
        # Block 2: 16 -> 32 channels
        self.block2 = torch.nn.Sequential(
            torch.nn.Conv2d(16, 32, 3, stride=2, padding=1),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(2, 2),
            torch.nn.Dropout(0.25),
        )
        
        # Block 3: 32 -> 64 channels
        self.block3 = torch.nn.Sequential(
            torch.nn.Conv2d(32, 64, 3, stride=2, padding=1),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(2, 2),
            torch.nn.Dropout(0.25),
        )
        
        # Global average pooling and classifier
        self.global_pool = torch.nn.AdaptiveAvgPool2d(1)
        self.linear_shrink = torch.nn.Linear(64, 8)
        self.linear_bottleneck = torch.nn.Sequential(*[FeedForwardWithSkip(8, 8) for _ in range(10)])
        self.classifier = torch.nn.Linear(8, num_classes)
    
    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.global_pool(x)
        x = x.view(x.size(0), -1)
        x = self.linear_shrink(x)
        x = self.linear_bottleneck(x)
        x = self.classifier(x)
        return x

In [4]:
model = ConvolutionWithSkipTumorClassification()
print(summary(model, input_size=(4, 1, 224, 224)))

Layer (type:depth-idx)                   Output Shape              Param #
ConvolutionWithSkipTumorClassification   [4, 4]                    --
├─Sequential: 1-1                        [4, 16, 56, 56]           --
│    └─Conv2d: 2-1                       [4, 16, 112, 112]         160
│    └─ReLU: 2-2                         [4, 16, 112, 112]         --
│    └─MaxPool2d: 2-3                    [4, 16, 56, 56]           --
│    └─Dropout: 2-4                      [4, 16, 56, 56]           --
├─Sequential: 1-2                        [4, 32, 14, 14]           --
│    └─Conv2d: 2-5                       [4, 32, 28, 28]           4,640
│    └─ReLU: 2-6                         [4, 32, 28, 28]           --
│    └─MaxPool2d: 2-7                    [4, 32, 14, 14]           --
│    └─Dropout: 2-8                      [4, 32, 14, 14]           --
├─Sequential: 1-3                        [4, 64, 3, 3]             --
│    └─Conv2d: 2-9                       [4, 64, 7, 7]             18,496
│    └─

In [4]:
from torch.utils.data import DataLoader
import os
import json

train_data = MultiTaskBrainDataset(
    os.path.join("..", "brisc2025"),
    mode="classification",
    train_or_test="train",
    subset="train",
    augment=True,
    seed=42
)

val_data = MultiTaskBrainDataset(
    os.path.join("..", "brisc2025"),
    mode="classification",
    train_or_test="train",
    subset="val",
    augment=False,
    seed=42
)

train_loader = DataLoader(train_data, batch_size=16, shuffle=True, num_workers=0)
val_loader = DataLoader(val_data, batch_size=16, shuffle=False, num_workers=0)

loss_fn = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters())

best_val_acc    = 0.0
min_improvement = 0.000
checkpoint_dir  = os.path.join("..", "results", "classification", "tumor_classification_v3_grant")
os.makedirs(checkpoint_dir, exist_ok=True)

best_model_path = os.path.join(checkpoint_dir, "best_model.pt")
metadata_path   = os.path.join(checkpoint_dir, "training_metadata.json")

training_metadata = {
    "best_val_acc": 0.0,
    "best_epoch":   None,
    "history":      []
}

for epoch in range(100):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        predictions = model(inputs)
        loss = loss_fn(predictions, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        preds = torch.argmax(predictions, dim=1)
        true_labels = torch.argmax(labels, dim=1)
        correct += (preds == true_labels).sum().item()
        total += labels.shape[0]

    epoch_loss = running_loss / len(train_loader)
    epoch_acc  = correct / total

    model.eval()
    val_loss    = 0.0
    val_correct = 0
    val_total   = 0

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            predictions = model(inputs)
            loss = loss_fn(predictions, labels)

            val_loss += loss.item()

            preds = torch.argmax(predictions, dim=1)
            true_labels = torch.argmax(labels, dim=1)
            val_correct += (preds == true_labels).sum().item()
            val_total   += labels.shape[0]

    val_loss /= len(val_loader)
    val_acc   = val_correct / val_total

    print(f"Epoch {epoch+1:03d} | Train Loss: {epoch_loss:.4f} | Train Acc: {epoch_acc:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

    # Record epoch metrics in metadata
    training_metadata["history"].append({
        "epoch":      epoch + 1,
        "train_loss": round(epoch_loss, 6),
        "train_acc":  round(epoch_acc,  6),
        "val_loss":   round(val_loss,   6),
        "val_acc":    round(val_acc,    6),
    })

    # Save best model and update metadata if improved
    if val_acc >= best_val_acc + min_improvement:
        best_val_acc = val_acc
        training_metadata["best_val_acc"] = round(val_acc, 6)
        training_metadata["best_epoch"]   = epoch + 1

        torch.save({
            "epoch":                epoch + 1,
            "model_state_dict":     model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "val_acc":              val_acc,
            "val_loss":             val_loss,
        }, best_model_path)
        print(f"  -> Best model saved (epoch {epoch+1}, val_acc={val_acc:.4f})")

    # Write metadata JSON after every epoch
    with open(metadata_path, "w") as f:
        json.dump(training_metadata, f, indent=4)

Epoch 001 | Train Loss: 1.1665 | Train Acc: 0.4645 | Val Loss: 0.9449 | Val Acc: 0.5840
  -> Best model saved (epoch 1, val_acc=0.5840)
Epoch 002 | Train Loss: 0.9076 | Train Acc: 0.5897 | Val Loss: 0.8166 | Val Acc: 0.6460
  -> Best model saved (epoch 2, val_acc=0.6460)
Epoch 003 | Train Loss: 0.8171 | Train Acc: 0.6365 | Val Loss: 0.7066 | Val Acc: 0.7350
  -> Best model saved (epoch 3, val_acc=0.7350)
Epoch 004 | Train Loss: 0.7669 | Train Acc: 0.6757 | Val Loss: 0.7117 | Val Acc: 0.6900
Epoch 005 | Train Loss: 0.7102 | Train Acc: 0.6980 | Val Loss: 0.6996 | Val Acc: 0.7210
Epoch 006 | Train Loss: 0.6758 | Train Acc: 0.7150 | Val Loss: 0.6529 | Val Acc: 0.7700
  -> Best model saved (epoch 6, val_acc=0.7700)
Epoch 007 | Train Loss: 0.6659 | Train Acc: 0.7238 | Val Loss: 0.6051 | Val Acc: 0.7710
  -> Best model saved (epoch 7, val_acc=0.7710)
Epoch 008 | Train Loss: 0.6199 | Train Acc: 0.7490 | Val Loss: 0.5854 | Val Acc: 0.7630
Epoch 009 | Train Loss: 0.6168 | Train Acc: 0.7548 | Val

In [4]:
import os
import numpy as np
import torch
from torch.utils.data import DataLoader
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix

device = torch.device("cpu")

CLASS_NAMES = ["glioma", "meningioma", "pituitary", "no_tumor"]

test_data = MultiTaskBrainDataset(
    os.path.join("..", "brisc2025"),
    mode="classification",
    train_or_test="test",
    augment=False,
    seed=42,
)

checkpoint_dir  = os.path.join("..", "results", "classification", "tumor_classification_v3_grant")
best_model_path = os.path.join(checkpoint_dir, "best_model.pt")
loss_fn = torch.nn.CrossEntropyLoss()
checkpoint = torch.load(best_model_path, map_location=device)
state_dict  = {k.replace("_orig_mod.", ""): v for k, v in checkpoint["model_state_dict"].items()}

model = ConvolutionWithSkipTumorClassification()
model.load_state_dict(state_dict)
model.to(device)
model.eval()
print(f"Loaded checkpoint from epoch {checkpoint['epoch']} (val_acc={checkpoint.get('val_acc', 'N/A')})")

loss_fn = torch.nn.CrossEntropyLoss()
loader  = DataLoader(test_data, batch_size=4, shuffle=False, num_workers=0)

test_loss    = 0.0
test_correct = 0
test_total   = 0
all_preds  = []
all_labels = []

with torch.no_grad():
    for inputs, labels in loader:
        inputs, labels = inputs.to(device), labels.to(device)
        predictions = model(inputs)

        test_loss += loss_fn(predictions, labels).item()

        preds       = torch.argmax(predictions, dim=1)
        true_labels = torch.argmax(labels, dim=1)

        test_correct += (preds == true_labels).sum().item()
        test_total   += labels.shape[0]

        all_preds.append(preds.cpu())
        all_labels.append(true_labels.cpu())

test_loss /= len(loader)
test_acc   = test_correct / test_total

all_preds  = torch.cat(all_preds).numpy()
all_labels = torch.cat(all_labels).numpy()

print(f"\nTest Loss : {test_loss:.4f}")
print(f"Test Acc  : {test_acc:.4f}\n")

# Per-class accuracy
print(f"{'Tumor Type':<22} {'N':>5} {'Correct':>8} {'Accuracy':>10} {'Precision':>10} {'Recall':>8} {'F1':>8}")
print("-" * 76)
for idx, name in enumerate(CLASS_NAMES):
    mask      = all_labels == idx
    n         = mask.sum()
    correct   = ((all_preds == idx) & mask).sum()
    acc       = correct / n if n > 0 else 0.0
    p  = precision_score(all_labels == idx, all_preds == idx, zero_division=0)
    r  = recall_score   (all_labels == idx, all_preds == idx, zero_division=0)
    f  = f1_score       (all_labels == idx, all_preds == idx, zero_division=0)
    print(f"{name:<22} {n:>5} {correct:>8} {acc:>10.4f} {p:>10.4f} {r:>8.4f} {f:>8.4f}")

print("-" * 76)
overall_p = precision_score(all_labels, all_preds, average="weighted", zero_division=0)
overall_r = recall_score   (all_labels, all_preds, average="weighted", zero_division=0)
overall_f = f1_score       (all_labels, all_preds, average="weighted", zero_division=0)
print(f"{'OVERALL':<22} {test_total:>5} {test_correct:>8} {test_acc:>10.4f} {overall_p:>10.4f} {overall_r:>8.4f} {overall_f:>8.4f}")

# Confusion matrix
cm = confusion_matrix(all_labels, all_preds)
print("\nConfusion matrix (rows = true, cols = predicted):")
print(f"{'':>12}", "  ".join(f"{n:>10}" for n in CLASS_NAMES))
for i, name in enumerate(CLASS_NAMES):
    print(f"{name:<12}", "  ".join(f"{cm[i, j]:>10}" for j in range(len(CLASS_NAMES))))

C:\Users\glawl\AppData\Local\Temp\ipykernel_7780\43040646.py:22: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(best_model_path, map_location=device)


Loaded checkpoint from epoch 98 (val_acc=0.933)

Test Loss : 0.2658
Test Acc  : 0.8930

Tumor Type                 N  Correct   Accuracy  Precision   Recall       F1
----------------------------------------------------------------------------
glioma                   254      210     0.8268     0.9502   0.8268   0.8842
meningioma               306      248     0.8105     0.8522   0.8105   0.8308
pituitary                300      298     0.9933     0.9490   0.9933   0.9707
no_tumor                 140      137     0.9786     0.7874   0.9786   0.8726
----------------------------------------------------------------------------
OVERALL                 1000      893     0.8930     0.8971   0.8930   0.8922

Confusion matrix (rows = true, cols = predicted):
                 glioma  meningioma   pituitary    no_tumor
glioma              210          39           5           0
meningioma           10         248          11          37
pituitary             0           2         298           0